# RAG Evaluation with RAGAS + Gemini (generation) + Groq (judge)

This notebook builds a small RAG pipeline (Qdrant + Gemini) and evaluates it with
[RAGAS](https://docs.ragas.io). **Generation** (the RAG pipeline that answers
questions) uses Google Gemini. The **RAGAS judge LLM** (the model that scores
Faithfulness / Response Relevancy) uses **Groq** instead of Gemini, because
Gemini's free tier enforces a very small daily request cap.

**Fixed in this pass (see Section 5 and Section 10 for the details):**
- **`'n' : number must be at most 1` from Groq** - root-caused and fixed.
  `ChatGroq` has an `n` attribute, and RAGAS's `LangchainLLMWrapper` sets
  `langchain_llm.n = strictness` directly on it before firing a request. Groq's
  Chat Completions API hard-caps `n=1` per request, no matter what value is
  sent. Fixed with `bypass_n=True` on the wrapper (makes RAGAS fire N separate
  single-completion calls instead of one call requesting N completions) +
  keeping `strictness=1` on `ResponseRelevancy` as a second layer of safety.
- **`TypeError: ...single_turn_ascore() got an unexpected keyword argument 'run_config'`**
  and **`TypeError: Faithfulness.__init__() got an unexpected keyword argument 'run_config'`**
  - `run_config` doesn't belong on the metric or on `single_turn_ascore()` in
  this ragas version. It belongs on the **LLM wrapper**:
  `LangchainLLMWrapper(chat_model, run_config=RunConfig(...))`. Fixed.
- **`NameError: name 'result' is not defined`** and the repeated `ClientError 429`
  on `rag_pipeline(...)` - this was Gemini's **daily** generation quota (20
  requests/day/model on the free tier) being exhausted by re-running the same
  generation cell 5 times while debugging. Consolidated back down to a single
  `result = rag_pipeline(...)` cell; see the note in Section 8 for how to avoid
  burning quota like this again.
- Consolidated the duplicated `ragas_response_relevancy` / `run_evaluations`
  definitions (there were 4 versions across the notebook, each fixing one
  symptom without fixing the underlying cause) into one final, working version.


## 1. Install dependencies

These versions were verified together in a clean virtual environment
(`pip check` reports **no broken requirements**):

| package | version |
|---|---|
| ragas | 0.3.7 |
| langchain | 0.3.30 |
| langchain-core | 0.3.86 |
| langchain-community | 0.3.31 |
| langchain-google-genai | 2.1.12 |
| langchain-groq | 0.3.8 |
| google-genai | latest 2.x |
| qdrant-client | latest 1.x |
| langsmith | latest |
| python-dotenv | latest |

> ⚠️ Do **not** `pip install -U langchain langchain-google-genai` on top of this.
> The newest `langchain-google-genai` (4.x) requires `langchain-core>=1.4`,
> which is incompatible with `ragas==0.3.7` (which still depends on the
> `langchain-community` 0.3.x line). Mixing the two is what caused the
> `ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'`
> style errors in the original notebook.
>
> `langchain-groq==0.3.8` needs `langchain-core>=0.3.75,<1.0.0`, which fits the
> pinned `langchain-core==0.3.86` above — no conflict with the rest of the stack.


In [1]:
! uv pip install -q "ragas==0.3.7" \
    "langchain==0.3.30" \
    "langchain-core==0.3.86" \
    "langchain-community==0.3.31" \
    "langchain-google-genai==2.1.12" \
    "langchain-groq==0.3.8" \
    "google-genai" \
    "qdrant-client" \
    "langsmith" \
    "python-dotenv" \
    "nest_asyncio"


## 2. Imports

In [2]:
import os
import asyncio

import nest_asyncio
from dotenv import load_dotenv

# Google & Qdrant
from google import genai
from google.genai import types
from qdrant_client import QdrantClient

# LangSmith
from langsmith import Client

# LangChain (Gemini) wrapper used by the RAG pipeline's generation step
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# LangChain (Groq) wrapper used ONLY as the RAGAS judge LLM
from langchain_groq import ChatGroq

# RAGAS
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    IDBasedContextPrecision,
    IDBasedContextRecall,
)

# Lets `await` work smoothly inside Jupyter regardless of the kernel's event loop
nest_asyncio.apply()


2026-07-10 22:55:20.661979667 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## 3. Environment & clients

Put a `.env` file next to this notebook containing:

```
GOOGLE_API_KEY=your-gemini-api-key
GROQ_API_KEY=your-groq-api-key
LANGSMITH_API_KEY=your-langsmith-api-key
```

`GOOGLE_API_KEY` is used for the RAG pipeline (generation + embeddings).
`GROQ_API_KEY` is used only for the RAGAS judge LLM (section 5 below).


In [3]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file!")

# Native google-genai client -> used for the RAG pipeline itself
# (embeddings + answer generation)
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# LangSmith client -> used to pull the evaluation dataset
ls_client = Client()


## 4. Shared model configuration

**Generation stays on Gemini. The RAGAS judge moves to Groq.** Two things worth knowing:

1. **Why the judge moved providers, not just models.** Sharing one model for
   generation *and* evaluation only has a mild "self-preference bias" (a
   model rates its own answers slightly more favorably). That's a quality
   concern, not the bug you hit — see Section 10 for what actually caused the
   persistent errors.
2. **Judge model note:** `llama-3.3-70b-versatile` is being deprecated by Groq
   (announced June 17, 2026). This notebook uses `openai/gpt-oss-120b`, Groq's
   current recommended general-purpose replacement. Check
   https://console.groq.com/docs/models for the latest lineup before relying
   on this in a long-lived project.
3. **Embeddings stay on Gemini.** Groq doesn't serve an embeddings endpoint,
   so `ResponseRelevancy` (which needs embeddings, not just an LLM) keeps
   using the same `EMBEDDING_MODEL` as Qdrant retrieval.
4. **`GROQ_RUN_CONFIG` below** controls retry/concurrency behavior for the
   Groq judge calls specifically. It must be passed into `LangchainLLMWrapper(...,
   run_config=...)` — not into the metric classes or into `single_turn_ascore()`,
   which don't accept it (that mismatch is what threw the two `TypeError`s).


In [4]:
# ---------------------------------------------------------------
# Single source of truth for every model name used in this notebook.
# ---------------------------------------------------------------
GENERATION_MODEL = "gemini-2.5-flash"     # used by the RAG pipeline to answer questions (Gemini)
EMBEDDING_MODEL = "gemini-embedding-001"  # used for BOTH Qdrant retrieval and RAGAS ResponseRelevancy
                                           # (models/embedding-001 and models/text-embedding-004
                                           # were retired by Google on Jan 14, 2026 - do not use them)

JUDGE_MODEL = "openai/gpt-oss-120b"       # used by RAGAS to score faithfulness/relevancy (Groq)
                                           # llama-3.3-70b-versatile is being deprecated by Groq -
                                           # check https://console.groq.com/docs/models if this stops working

# Retry/concurrency behavior for the Groq judge LLM specifically. This goes on
# the LLM wrapper (Section 5) - NOT on the metric classes and NOT on
# single_turn_ascore(), neither of which accept a run_config argument.
from ragas.run_config import RunConfig
GROQ_RUN_CONFIG = RunConfig(max_workers=1, max_retries=5, max_wait=30)


## 5. RAGAS evaluator LLM & embeddings


In [5]:
# RAGAS needs its own LLM (to judge faithfulness / relevancy) and its own
# embedding model (for ResponseRelevancy).
#
# - Judge LLM -> Groq (ChatGroq), not Gemini.
# - `run_config=GROQ_RUN_CONFIG` -> runs the judge sequentially (max_workers=1)
#   with its own retry/backoff, independent of Gemini's limits.
# - `bypass_n=True` -> THIS is what actually fixes the
#   `'n' : number must be at most 1` error from Groq. RAGAS's LangchainLLMWrapper
#   normally does `langchain_llm.n = <requested completions>` and fires ONE API
#   call asking for N completions at once. Groq's Chat Completions API hard-caps
#   n=1 per call and rejects anything else - so any metric that ever asks for
#   more than 1 completion (e.g. ResponseRelevancy's `strictness`) will always
#   400 against Groq, regardless of what strictness is set to. `bypass_n=True`
#   makes RAGAS fire N separate single-completion calls instead of one call
#   requesting N completions, so it can never send n>1 to Groq.
# - Embeddings -> stay on Gemini, since Groq has no embeddings endpoint.
ragas_llm = LangchainLLMWrapper(
    ChatGroq(model=JUDGE_MODEL, groq_api_key=GROQ_API_KEY),
    run_config=GROQ_RUN_CONFIG,
    bypass_n=True,
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model=f"models/{EMBEDDING_MODEL}", google_api_key=GOOGLE_API_KEY)
)


/tmp/ipykernel_2519478/894051878.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  ragas_llm = LangchainLLMWrapper(
/tmp/ipykernel_2519478/894051878.py:23: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


## 6. Download a reference example from LangSmith


In [6]:
dataset = ls_client.read_dataset(dataset_name="rag-evaluation-dataset")

# Pull the examples ONCE and reuse them (the original notebook called
# list_examples three separate times for no reason)
examples = list(ls_client.list_examples(dataset_id=dataset.id, limit=10))

reference_input = examples[0].inputs
reference_output = examples[0].outputs

reference_input, reference_output


({'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?'},
 {'ground_truth': "I'm sorry, but the specific duration of the warranty for the Garmin 890 RV GPS Navigator is not mentioned in the product description, only that it comes with a 'Limited Warranty'.",
  'reference_context_ids': [],
  'reference_descriptions': []})

## 7. RAG pipeline

* `get_embedding` – embeds a query with the shared `EMBEDDING_MODEL`
* `retrieve_data` – queries Qdrant and returns ids/context/ratings/scores
* `process_context` – turns retrieved chunks into a formatted context string
* `build_prompt` – builds the final prompt for the answer-generation model
* `generate_answer` – calls Gemini (`GENERATION_MODEL`) to answer the question
* `rag_pipeline` – glues everything together


In [7]:
def get_embedding(text, model=EMBEDDING_MODEL):
    """Generate an embedding vector for `text` using a Gemini embedding model."""
    response = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(),
    )
    return response.embeddings[0].values


def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-03",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):
    formatted_context = ""
    for id_, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id_}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    """NOTE: this function was left incomplete (missing `:` and body) in the
    original notebook, which raised a SyntaxError. Fixed here."""
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use the word "context" and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt


def generate_answer(prompt, model_name=GENERATION_MODEL):
    """Generates a response using the specified Gemini model."""
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    return response.text


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }


### Quick smoke test

In [8]:
rag_pipeline("Can I get some charger?", top_k=5)


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

## 8. Run the pipeline on the LangSmith reference question

**Careful re-running this cell.** `generate_answer()` calls Gemini's
`generateContent`, which is metered by the same free-tier daily cap
(`GenerateRequestsPerDayPerProjectPerModel-FreeTier`, currently 20/day for
`gemini-2.5-flash` on this project) discussed earlier. Re-running this cell
repeatedly while debugging (as happened here — it was run 5 times back to
back) burns that daily budget on generation alone, before evaluation even
starts. Run it once, inspect `result`, and only re-run if you actually need a
fresh answer. If you're iterating a lot, either enable billing on the Google
Cloud project (removes the daily cap) or cache `result` locally instead of
regenerating it each time.


In [ ]:
result = rag_pipeline(reference_input["question"])
result


## 9. RAGAS metrics

Four metrics are used:

- **Faithfulness** – is the answer grounded in the retrieved context? (needs an LLM — Groq)
- **ResponseRelevancy** – is the answer relevant to the question? (needs an LLM — Groq — and embeddings — Gemini)
- **IDBasedContextPrecision** – of the retrieved ids, how many are in the reference ids? (no LLM needed, pure ID comparison)
- **IDBasedContextRecall** – of the reference ids, how many were retrieved? (no LLM needed, pure ID comparison)


In [ ]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)


async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    # strictness=1 -> only 1 candidate question generated (default is 3).
    # Combined with bypass_n=True on ragas_llm (Section 5), this guarantees
    # Groq never receives a request asking for more than 1 completion, which
    # is what caused the earlier "'n' : number must be at most 1" error.
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings, strictness=1)
    return await scorer.single_turn_ascore(sample)


async def ragas_context_precision_id_based(run, example):
    # No LLM call - pure ID set comparison, free to run as often as you like.
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_id_based(run, example):
    # No LLM call - pure ID set comparison, free to run as often as you like.
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)


## 10. Run all evaluations

### What actually caused the errors, and what fixed each one

| Error you saw | Real cause | Fix |
|---|---|---|
| `ClientError 429 RESOURCE_EXHAUSTED` on `rag_pipeline(...)` | Gemini's **daily** free-tier cap for `generateContent` (20 req/day/model on this project) — re-running the generation cell repeatedly burned it | Deduplicated to a single run in Section 8; enable billing if you need to iterate a lot |
| `Retrying langchain_google_genai...ResourceExhausted` while "Evaluating Faithfulness" | At that point in the session, `ragas_llm` was still bound to a `ChatGoogleGenerativeAI` instance from an earlier, out-of-order cell run — Section 5 hadn't actually been re-run with the Groq version in that kernel session | Use **Kernel → Restart & Run All** after making changes like this, instead of re-running cells out of order — stale variable bindings from earlier edits are a classic Jupyter footgun |
| `BadRequestError ... "'n' : number must be at most 1"` | `ChatGroq` has an `n` attribute; RAGAS's `LangchainLLMWrapper` sets `langchain_llm.n = strictness` and fires one request asking for that many completions. Groq's API hard-caps `n=1` and rejects anything else, regardless of what `strictness` is set to | `bypass_n=True` on the wrapper (Section 5) — makes RAGAS fire N separate single-completion calls instead of one N-completion call |
| `TypeError: single_turn_ascore() got an unexpected keyword argument 'run_config'` / `TypeError: Faithfulness.__init__() got an unexpected keyword argument 'run_config'` | `run_config` isn't a parameter of the metric classes or of `single_turn_ascore()` in this ragas version | `run_config` goes on `LangchainLLMWrapper(chat_model, run_config=...)` instead (Section 5) |
| `NameError: name 'result' is not defined` | A prior `rag_pipeline(...)` call had failed with the 429 above, so `result` was never assigned before this cell ran | Fixed by running Section 8 once, successfully, before this section |

The two ID-based metrics never call an LLM at all, so none of the above
applies to them — they'll always succeed regardless of Groq/Gemini quota
state.


In [ ]:
async def run_evaluations(run, example):
    print("Evaluating Faithfulness...")
    faithfulness_score = await ragas_faithfulness(run, example)

    print("Evaluating Response Relevancy...")
    relevancy_score = await ragas_response_relevancy(run, example)

    print("Evaluating ID-Based Precision & Recall...")
    precision_score = await ragas_context_precision_id_based(run, example)
    recall_score = await ragas_context_recall_id_based(run, example)

    return {
        "faithfulness": faithfulness_score,
        "response_relevancy": relevancy_score,
        "id_based_context_precision": precision_score,
        "id_based_context_recall": recall_score,
    }


scores = await run_evaluations(result, reference_output)
print("\nFinal Evaluation Scores:")
print(scores)
